In [21]:
import os
import sys

In [23]:
python_path = sys.executable
python_path

'c:\\Users\\domin\\OneDrive\\Pulpit\\d_eg\\pyspark_sqlalchemy_postgres\\Pyspark_test\\.venv\\Scripts\\python.exe'

In [24]:
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [25]:
spark = (
    SparkSession.builder.appName('Pyspark_intro')
    .master('local[*]')
    .config('spark.pyspark.python', python_path)
    .config('spark.pyspark.driver.python', python_path)
    .config('spark.python.use.daemon', 'false')
    .config('spark.python.worker.faulthandler.enabled', 'true')
    .getOrCreate()
)

In [26]:
d = [('Jan', 'Ksiegowosc', 9000),
     ('Tomasz', 'Ksiegowosc', 9000),
     ('Karolina', 'HR', 8000)]
c = ['Imie', 'Dzial', 'Pensja']

df = spark.createDataFrame(d, c)

In [27]:
d_filtered = df.filter(F.col('Pensja') > 8500).groupBy('Dzial').agg(F.avg('Pensja'))

In [28]:
d_filtered.explain(True)

== Parsed Logical Plan ==
'Aggregate ['Dzial], ['Dzial, unresolvedalias('avg('Pensja))]
+- Filter (Pensja#65L > cast(8500 as bigint))
   +- LogicalRDD [Imie#63, Dzial#64, Pensja#65L], false

== Analyzed Logical Plan ==
Dzial: string, avg(Pensja): double
Aggregate [Dzial#64], [Dzial#64, avg(Pensja#65L) AS avg(Pensja)#70]
+- Filter (Pensja#65L > cast(8500 as bigint))
   +- LogicalRDD [Imie#63, Dzial#64, Pensja#65L], false

== Optimized Logical Plan ==
Aggregate [Dzial#64], [Dzial#64, avg(Pensja#65L) AS avg(Pensja)#70]
+- Project [Dzial#64, Pensja#65L]
   +- Filter (isnotnull(Pensja#65L) AND (Pensja#65L > 8500))
      +- LogicalRDD [Imie#63, Dzial#64, Pensja#65L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Dzial#64], functions=[avg(Pensja#65L)], output=[Dzial#64, avg(Pensja)#70])
   +- Exchange hashpartitioning(Dzial#64, 200), ENSURE_REQUIREMENTS, [plan_id=211]
      +- HashAggregate(keys=[Dzial#64], functions=[partial_avg(Pensja#65L)], output=[D

In [29]:
d_filtered.show()

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "C:\Users\domin\OneDrive\Pulpit\d_eg\pyspark_sqlalchemy_postgres\Pyspark_test\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3569, in main
    check_python_version(infile)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "C:\Users\domin\OneDrive\Pulpit\d_eg\pyspark_sqlalchemy_postgres\Pyspark_test\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker_util.py", line 86, in check_python_version
    raise PySparkRuntimeError(
    ...<5 lines>...
    )
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version: 3.13 than that in driver: 3.10, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.